In [1]:
!nvidia-smi

Mon Apr 13 20:07:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!pip uninstall -y transformers peft sentence-transformers

Found existing installation: transformers 4.36.2
Uninstalling transformers-4.36.2:
  Successfully uninstalled transformers-4.36.2
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: sentence-transformers 5.3.0
Uninstalling sentence-transformers-5.3.0:
  Successfully uninstalled sentence-transformers-5.3.0


In [7]:
!pip install -q transformers==4.41.2
!pip install -q peft==0.10.0
!pip install -q accelerate bitsandbytes sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 15.9 MB/s eta 0:00:00


In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
!git clone https://github.com/wentaoyuan/RoboPoint.git

fatal: destination path 'RoboPoint' already exists and is not an empty directory.


In [2]:
import sys
sys.path.insert(0, "/content/RoboPoint")

print("Path set")

Path set


In [3]:
import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [4]:
import sys
sys.path.insert(0, "/content/RoboPoint")

import torch
torch.set_default_dtype(torch.float16)

from robopoint.model.builder import load_pretrained_model
from robopoint.mm_utils import get_model_name_from_path

MODEL_PATH = "wentao-yuan/robopoint-v1-vicuna-v1.5-7b-lora"
MODEL_BASE = "lmsys/vicuna-7b-v1.5"

device = "cuda"

print("Loading model...")

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=MODEL_PATH,
    model_base=MODEL_BASE,
    model_name=get_model_name_from_path(MODEL_PATH),
    device_map="auto",
    device=device
)

print("Model loaded successfully")
print(context_len)

Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading LLaVA from base model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlavaLlamaForCausalLM were not initialized from the model checkpoint at lmsys/vicuna-7b-v1.5 and are newly initialized: ['model.mm_projector.0.bias', 'model.mm_projector.0.weight', 'model.mm_projector.2.bias', 'model.mm_projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `

...

adapter_model.safetensors:   0%|          | 0.00/640M [00:00<?, ?B/s]

Merging LoRA weights...
Model is loaded...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

✅ Model loaded successfully
2048


In [5]:
from google.colab import files

uploaded = files.upload()  # choose robopoint_input.zip

Saving robopoint_input.zip to robopoint_input.zip


In [6]:
import zipfile
import os

zip_name = list(uploaded.keys())[0]

extract_path = "robopoint_input"

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_path)

print("Extracted to:", extract_path)

Extracted to: robopoint_input


In [7]:
for root, dirs, files_list in os.walk(extract_path):
    level = root.replace(extract_path, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files_list:
        print(f"{indent}  {f}")

robopoint_input/
  outputs/
    camera_metadata.json
    prompts.jsonl
    renders/
      mugs/
        10c2b3eac377b9084b3c42e318f3affc/
          az_045.png
          az_180.png
          az_090.png
          az_000.png
          az_135.png
        1305b9266d38eb4d9f818dd0aa1a251/
          az_045.png
          az_180.png
          az_090.png
          az_000.png
          az_135.png


In [8]:
import json
import glob

prompt_files = glob.glob(extract_path + "/**/*.jsonl", recursive=True)
print("Found prompt files:", prompt_files)

prompt_path = prompt_files[0]

prompts = []
with open(prompt_path, "r") as f:
    for line in f:
        prompts.append(json.loads(line))

print("Num prompts:", len(prompts))
print(prompts[0])

Found prompt files: ['robopoint_input/outputs/prompts.jsonl']
Num prompts: 10
{'question_id': 'mug_1305b926_az000', 'image': 'mugs/1305b9266d38eb4d9f818dd0aa1a251/az_000.png', 'text': 'Point to where you would grasp the handle of the mug. Your answer should be formatted as a list of tuples, i.e. [(x1, y1), (x2, y2), ...], where each tuple contains the x and y coordinates of a point. The coordinates should be between 0 and 1, indicating the normalized pixel locations in the image.'}


In [9]:
from PIL import Image

image_sets = {}

image_root = extract_path

image_paths = glob.glob(image_root + "/**/*.png", recursive=True)

for p in image_paths:
    obj_id = p.split("/")[-2]  # folder = object id
    image_sets.setdefault(obj_id, []).append(p)

print("Num objects:", len(image_sets))
print("Example object:", list(image_sets.keys())[0])

Num objects: 2
Example object: 10c2b3eac377b9084b3c42e318f3affc


In [10]:
def load_object_images(obj_id):
    paths = sorted(image_sets[obj_id])
    imgs = [Image.open(p).convert("RGB") for p in paths]
    return imgs, paths

In [12]:
from robopoint.constants import IMAGE_TOKEN_INDEX
from robopoint.conversation import conv_templates

conv = conv_templates["vicuna_v1"].copy()

question = prompt.get("prompt", prompt.get("text", "Describe the image."))

conv.append_message(conv.roles[0], question)
conv.append_message(conv.roles[1], None)

prompt_text = conv.get_prompt()

In [19]:
import os
import json
import torch
from PIL import Image

# from llava.mm_utils import process_images, tokenizer_image_token
# from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
# from llava.conversation import conv_templates

from robopoint.mm_utils import process_images, tokenizer_image_token
from robopoint.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from robopoint.conversation import conv_templates

IMAGE_ROOT = "robopoint_input/outputs/renders"
PROMPTS_FILE = "robopoint_input/outputs/prompts.jsonl"
OUTPUT_FILE = "predictions.jsonl"

# Load prompts
prompts = []
with open(PROMPTS_FILE) as f:
    for line in f:
        prompts.append(json.loads(line.strip()))

print(f"Loaded {len(prompts)} prompts")

results = []

for entry in prompts:
    question_id = entry["question_id"]
    image_path = os.path.join(IMAGE_ROOT, entry["image"])
    text = entry["text"]

    print(f"\nProcessing: {question_id}")

    # Load image
    image = Image.open(image_path).convert("RGB")

    image_tensor = process_images([image], image_processor, model.config)
    image_tensor = image_tensor.to(model.device, dtype=torch.float16)

    # IMPORTANT: image sizes (fixes many LLaVA/RoboPoint bugs)
    image_sizes = torch.tensor(
        [[image.size[1], image.size[0]]],
        dtype=torch.long,
        device=model.device
    )

    # Build conversation
    conv = conv_templates["vicuna_v1"].copy()
    inp = DEFAULT_IMAGE_TOKEN + "\n" + text
    conv.append_message(conv.roles[0], inp)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()

    # Tokenize
    input_ids = tokenizer_image_token(
        prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt"
    ).unsqueeze(0).to(model.device)

    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            images=image_tensor,
            image_sizes=image_sizes,
            do_sample=False,
            max_new_tokens=128,
            use_cache=True
        )

    # Decode (safe version)
    output = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

    print(f"  Prediction: {output}")

    results.append({
        "question_id": question_id,
        "image": entry["image"],
        "prediction": output
    })

# Save results
with open(OUTPUT_FILE, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

print(f"\nDone — saved {len(results)} predictions to {OUTPUT_FILE}")

Loaded 10 prompts

Processing: mug_1305b926_az000


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  Prediction: [(0.531, 0.458), (0.548, 0.456), (0.516, 0.460)]

Processing: mug_1305b926_az045
  Prediction: [(0.303, 0.406), (0.338, 0.406), (0.355, 0.404), (0.320, 0.404)]

Processing: mug_1305b926_az090
  Prediction: [(0.408, 0.406), (0.439, 0.406), (0.464, 0.406), (0.500, 0.406), (0.522, 0.406), (0.483, 0.404)]

Processing: mug_1305b926_az135
  Prediction: [(0.469, 0.490), (0.453, 0.479), (0.484, 0.479)]

Processing: mug_1305b926_az180
  Prediction: [(0.480, 0.506), (0.481, 0.469), (0.483, 0.531), (0.483, 0.442), (0.483, 0.554)]

Processing: mug_10c2b3ea_az000
  Prediction: [(0.539, 0.471), (0.555, 0.481), (0.523, 0.465)]

Processing: mug_10c2b3ea_az045
  Prediction: [(0.539, 0.477), (0.555, 0.467), (0.523, 0.469), (0.570, 0.456)]

Processing: mug_10c2b3ea_az090
  Prediction: [(0.489, 0.398), (0.505, 0.410), (0.520, 0.400)]

Processing: mug_10c2b3ea_az135
  Prediction: [(0.459, 0.458), (0.480, 0.456), (0.444, 0.448)]

Processing: mug_10c2b3ea_az180
  Prediction: [(0.450, 0.448), (0

In [20]:
from google.colab import files
files.download("predictions.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>